# Reconocimiento de frutas con una red CNN

Clasificador de **33 tipos de frutas** con una red neuronal convolucional (CNN) completa.

**Pipeline:**
1. Leer el dataset (`dataset.zip` -> carpeta `fruits/`).
2. Convertir cada imagen a escala de grises `88x88` y aplanarla (7744 píxeles).
3. Construir la CNN (Conv + MaxPool + capas densas).
4. Entrenar, evaluar y mostrar predicciones.

> Kernel recomendado: **Python (ClaseIa TF)** o Google Colab con GPU.


In [ ]:
import os
import zipfile
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Input
from sklearn.model_selection import train_test_split

tf.get_logger().setLevel("ERROR")

ZIP_PATH   = "dataset.zip"
CSV_GRIS   = "frutas_gris.csv"
EXTRACCION = "dataset_extracted"
ANCHO, ALTO = 88, 88
PIXELES    = ANCHO * ALTO
NUM_CLASES = 33
ETIQUETA   = str(PIXELES)   # columna "7744"

# Lector de imágenes en escala de grises: cv2 si existe, si no TensorFlow.
try:
    import cv2
    def leer_gris(datos):
        arr = np.frombuffer(datos, np.uint8)
        img = cv2.imdecode(arr, cv2.IMREAD_GRAYSCALE)
        return None if img is None else cv2.resize(img, (ANCHO, ALTO))
    print("Usando OpenCV para leer imágenes.")
except ImportError:
    def leer_gris(datos):
        img = tf.io.decode_image(datos, channels=1, expand_animations=False)
        img = tf.image.resize(img, [ALTO, ANCHO])
        return np.asarray(img, dtype=np.uint8)[:, :, 0]
    print("Usando TensorFlow para leer imágenes.")


In [ ]:
# ============================================================
# 1) Construir / cargar el dataset en escala de grises
# ============================================================
def construir_desde_zip():
    if not os.path.isdir(os.path.join(EXTRACCION, "fruits")):
        with zipfile.ZipFile(ZIP_PATH, "r") as z:
            z.extractall(EXTRACCION)
    ruta_fruits = os.path.join(EXTRACCION, "fruits")
    categorias = sorted(c for c in os.listdir(ruta_fruits)
                        if os.path.isdir(os.path.join(ruta_fruits, c)))
    mapeo = {cat: i for i, cat in enumerate(categorias)}
    print(f"{len(categorias)} categorías:", list(mapeo))

    imagenes = []
    for cat in categorias:
        ruta_cat = os.path.join(ruta_fruits, cat)
        for nombre in sorted(os.listdir(ruta_cat)):
            if nombre.lower().endswith((".jpg", ".jpeg", ".png")):
                imagenes.append((cat, os.path.join(ruta_cat, nombre)))

    pixeles = np.empty((len(imagenes), PIXELES), dtype=np.uint8)
    etiquetas = np.empty((len(imagenes),), dtype=np.int64)
    fila = 0
    for cat, ruta in imagenes:
        with open(ruta, "rb") as fh:
            img = leer_gris(fh.read())
        if img is None:
            continue
        pixeles[fila] = img.reshape(-1)     # gris 88x88 -> 7744 enteros 0-255
        etiquetas[fila] = mapeo[cat]
        fila += 1
    pixeles, etiquetas = pixeles[:fila], etiquetas[:fila]

    df = pd.DataFrame(pixeles, columns=[str(i) for i in range(PIXELES)])
    df[ETIQUETA] = etiquetas
    df.to_csv(CSV_GRIS, index=False)        # encabezado 0..7744, sin letras
    print("CSV guardado:", CSV_GRIS, df.shape)
    return df

if os.path.exists(CSV_GRIS):
    df = pd.read_csv(CSV_GRIS)
    print("Dataset cargado desde CSV:", df.shape)
elif os.path.exists(ZIP_PATH):
    df = construir_desde_zip()
else:
    raise FileNotFoundError("No se encontró 'frutas_gris.csv' ni 'dataset.zip'.")

display(df.head())


In [ ]:
# ============================================================
# 2) Preparar características (X) y etiquetas (y)
# ============================================================
df = df.dropna()

X = df.drop(columns=[ETIQUETA]).values.astype("float32") / 255.0
y = df[ETIQUETA].values.astype("int64")

# 7744 píxeles -> imagen 88x88 en escala de grises
X = X.reshape(-1, ANCHO, ALTO, 1)

print("X:", X.shape, X.dtype)
print("y:", y.shape, "| clases distintas:", len(np.unique(y)))


In [ ]:
# ============================================================
# 3) División del dataset (entrenamiento / prueba)
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Entrenamiento:", X_train.shape, "| Prueba:", X_test.shape)


In [ ]:
# ============================================================
# 4) Red convolucional completa
# ============================================================
modelo = Sequential([
    Input(shape=(ANCHO, ALTO, 1)),

    Conv2D(32, (3, 3), activation="relu"),   # bordes y líneas
    MaxPooling2D((2, 2)),

    Conv2D(64, (5, 5), activation="relu"),   # patrones de mayor tamaño
    MaxPooling2D((2, 2)),

    Conv2D(128, (3, 3), activation="relu"),  # características complejas
    MaxPooling2D((2, 2)),

    Flatten(),
    Dense(256, activation="relu"),
    Dense(128, activation="relu"),
    Dense(64, activation="relu"),

    Dense(NUM_CLASES)                        # 33 frutas (sin softmax = logits)
])

modelo.summary()

modelo.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)


In [ ]:
# ============================================================
# 5) Entrenamiento
# ============================================================
historial = modelo.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.20
)


In [ ]:
# ============================================================
# 6) Evaluación y predicciones
# ============================================================
loss, accuracy = modelo.evaluate(X_test, y_test, verbose=0)
print(f"Precisión: {accuracy * 100:.2f}%")

salidas = modelo.predict(X_test, verbose=0)

for i in range(10):
    print(f"Imagen {i + 1}: predicha={np.argmax(salidas[i])}  real={y_test[i]}")

modelo.save("clasificador_frutas.keras")
print("Modelo guardado: clasificador_frutas.keras")
